In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data whic

In [43]:
from biked_commons.prediction import loaders
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
model_path = models_and_scalers_path("validity_model.pt")
scaler_path = models_and_scalers_path("validity_scaler.pt")
preprocessor = Preprocessor(scaler_path=scaler_path, preprocess_fn=None, device=device)
converter = framed.clip_to_framed_tensor_builder(ordered_columns.ORDERED_COLUMNS, framed.FRAMED_ORDERED_COLUMNS)
model = torch.load(model_path, weights_only=False).to(device)

data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)
data_tens = torch.tensor(data.values, dtype=dtype).to(device)

data_framed, _ = loaders.load_validity()
framed_tensor = torch.tensor(data_framed.values.astype(float), dtype=dtype).to(device)

regen_framed_tensor = converter(data_tens)
regen_framed_tensor = regen_framed_tensor.to(device, dtype=dtype)


preprocessed = preprocessor(framed_tensor)
predictions = model(preprocessed)
validity = predictions-0.5

In [48]:
regen_framed_tensor

tensor([[1.0000, 0.0000, 0.0000,  ..., 0.0010, 0.0011, 0.6593],
        [0.0000, 1.0000, 0.0000,  ..., 0.0010, 0.0011, 0.5722],
        [1.0000, 0.0000, 0.0000,  ..., 0.0010, 0.0011, 0.6471],
        ...,
        [1.0000, 0.0000, 0.0000,  ..., 0.0010, 0.0011, 0.5477],
        [1.0000, 0.0000, 0.0000,  ..., 0.0010, 0.0011, 0.5422],
        [1.0000, 0.0000, 0.0000,  ..., 0.0010, 0.0011, 0.6550]],
       device='cuda:0')

In [41]:
data_framed

,Material=Steel,Material=Aluminum,Material=Titanium,SSB_Include,CSB_Include,CS Length,BB Drop,Stack,SS E,ST Angle,...,CSB Offset,SS Z,SS Thickness,CS Thickness,TT Thickness,BB Thickness,HT Thickness,ST Thickness,DT Thickness,DT Length
9237,True,False,False,0.0,0.0,0.454210,4.974500e-02,0.56541,0.028226,74.602428,...,0.381567,0.010334,0.006401,0.001903,0.003043,0.003054,0.008850,0.002345,0.006206,0.660599
8922,True,False,False,1.0,1.0,0.428072,7.297700e-02,0.56664,0.034613,72.058553,...,0.299777,0.008548,0.006327,0.002624,0.002506,0.000540,0.002825,0.000872,0.003573,0.674965
11585,False,True,False,1.0,1.0,0.358010,-2.328000e-02,0.55498,0.175450,74.661000,...,0.349621,0.008206,0.003124,0.006949,0.000855,0.005617,0.001092,0.001054,0.004003,0.568519
6892,False,False,True,1.0,1.0,0.404602,7.005700e-02,0.56557,0.044059,74.011709,...,0.350116,0.008996,0.001073,0.001192,0.000751,0.000813,0.000962,0.000798,0.000956,0.658852
4783,False,True,False,1.0,1.0,0.297080,-1.282000e-02,0.56564,0.053940,66.369610,...,0.300662,0.009216,0.001014,0.001998,0.001276,0.000651,0.008124,0.008348,0.005268,0.415527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7997,True,False,False,0.0,0.0,0.425420,2.526700e-02,0.56560,0.045000,72.488558,...,0.300000,0.009000,0.003461,0.001703,0.003410,0.001102,0.004535,0.002055,0.009791,0.674373
13948,True,False,False,0.0,0.0,0.409670,5.098800e-02,0.56523,0.039210,73.530997,...,0.290824,0.008977,0.002399,0.002827,0.005921,0.002497,0.003042,0.006572,0.002998,0.665984
1307,True,False,False,0.0,0.0,0.362440,1.989520e-15,0.56560,0.045000,74.000000,...,0.300000,0.009000,0.002200,0.002957,0.001814,0.000863,0.001952,0.003190,0.002940,0.570119
8870,False,False,True,1.0,1.0,0.392032,6.675600e-02,0.53600,0.173262,75.240875,...,0.349996,0.008863,0.001042,0.000980,0.003368,0.003429,0.001838,0.003297,0.003799,0.645932


In [32]:
np.max(predictions.cpu().detach().numpy())

np.float32(0.0)

In [11]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [12]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [13]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [14]:
eval_scores = evaluator(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [15]:
#check gradient of eval scores wrt data_tens
data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
eval_scores_sum = eval_scores.sum()
eval_scores_sum.backward()
print(data_tens.grad.shape)
print(data_tens.grad[0])



c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


torch.Size([4512, 97])
tensor([-2.5034e+00,  3.2444e+00,  1.8290e+00,  8.2056e+00, -3.8640e+00,
         6.7613e-01, -6.7381e-01, -5.1584e+00, -2.5499e+00, -9.9719e-01,
        -2.3199e-02,  9.6363e-03,  8.6363e-02,  4.4697e-02,  1.5241e-02,
         4.4005e-02,  2.5426e-03,  9.9909e-01,  5.8016e-04,  3.8800e+00,
         2.3072e-04,  1.9989e-03,  1.3424e-02, -6.5445e-02,  8.7754e-02,
        -1.3766e-03,  1.1892e-02,  5.6734e-02,  2.8252e-02,  9.0985e-02,
         2.6884e-01,  1.5586e-01, -1.2939e-02,  0.0000e+00,  5.0000e-01,
         0.0000e+00,  5.0000e-01,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         5.0409e-03,  2.3941e-03,  0.0000e+00,  1.5713e-03,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00, -1.0000e+00,
        -1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00, -5.3652e-01,  0.0000e+00, -1.0000e+00, -1.4543e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  7.1324e-01, -4.8394e-01,
         0.0000e+00,  0.0000

In [16]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [17]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

In [18]:
main_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
detailed_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Min Objective Score: Usability Score - 0 to 1                                                                 0.791411
Min Objective Score: Drag Force                                                                              27.965340
Min Objective Score: Knee Angle Error                                                                       186.338130
Min Objective Score: Hip Angle Error                                                                        809.888700
Min Objective Score: Arm Angle Error                                                                        848.619570
Min Objective Score: Mass                                                                                    22.190498
Min Objective Score: Planar Compliance                                                                      180.326250
Min Objective Score: Transverse Compliance                                                                  265.009770
Min Objective Score: Eccentric Compliance       